In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ustawienie stylu wykresów (będą dobrze wyglądać w sprawozdaniu)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Wczytanie zbioru danych (ścieżka względna z folderu notebooks/)
df = pd.read_csv("../data/tabular/healthcare-dataset-stroke-data.csv")

print(f"Wymiary zbioru: {df.shape[0]} wierszy, {df.shape[1]} kolumn")
display(df.head())

In [ ]:
print("--- Informacje o kolumnach i typach danych ---")
df.info()

print("\n--- Liczba brakujących wartości (NaN) ---")
missing = df.isnull().sum()
print(missing[missing > 0])

print(f"\nLiczba zduplikowanych wierszy: {df.duplicated().sum()}")

In [ ]:
# Statystyki dla cech numerycznych (bez ID)
display(df.drop(columns=["id"]).describe().round(2))

# Rozkład wartości dla zmiennych tekstowych (kategorycznych)
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
for col in cat_cols:
    print(f"\n--- Rozkład kategorii: {col} ---")
    print(df[col].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Rozkład zmiennej docelowej (stroke)
sns.countplot(data=df, x="stroke", ax=axes[0], palette="Set2")
axes[0].set_title("Rozkład zmiennej docelowej (Udar: 0 = Nie, 1 = Tak)")
for container in axes[0].containers:
    axes[0].bar_label(container)

# 2. Rozkład wieku względem wystąpienia udaru
sns.histplot(data=df, x="age", hue="stroke", kde=True, bins=30, ax=axes[1], palette="Set1")
axes[1].set_title("Rozkład wieku pacjentów vs Udar")

# 3. Rozkład poziomu glukozy (wykres pudełkowy - wyłapuje wartości odstające)
sns.boxplot(data=df, x="stroke", y="avg_glucose_level", ax=axes[2], palette="Set2")
axes[2].set_title("Poziom glukozy vs Udar (wartości odstające)")

plt.tight_layout()
plt.show()

# 4. Macierz korelacji zmiennych numerycznych
plt.figure(figsize=(8, 6))
numeric_df = df.drop(columns=["id"]).select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Macierz korelacji zmiennych numerycznych")
plt.show()

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Kopia danych i usunięcie kolumny 'id'
df_clean = df.drop(columns=["id"]).copy()

# 2. Uzupełnienie braków w 'bmi' medianą
df_clean["bmi"] = df_clean["bmi"].fillna(df_clean["bmi"].median())

# 3. Usunięcie pojedynczego rekordu 'Other' w 'gender' (dla czystości kodowania)
df_clean = df_clean[df_clean["gender"] != "Other"]

# 4. Zamiana zmiennych kategorycznych na liczby (One-Hot Encoding)
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True, dtype=int)

print(f"Wymiary danych po kodowaniu: {df_encoded.shape[0]} wierszy, {df_encoded.shape[1]} kolumn")

# 5. Podział na cechy (X) i zmienną docelową (y)
X = df_encoded.drop(columns=["stroke"])
y = df_encoded["stroke"]

# 6. Podział na zbiór treningowy (80%) i testowy (20%) z zachowaniem proporcji klas (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Standaryzacja cech numerycznych (dopasowana TYLKO do zbioru treningowego!)
scaler = StandardScaler()
num_cols = ["age", "avg_glucose_level", "bmi"]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Zbiór treningowy: {X_train.shape}, Zbiór testowy: {X_test.shape}")
print(f"Liczba udarów w zbiorze testowym: {y_test.sum()} z {len(y_test)} pacjentów")

Wymiary danych po kodowaniu: 5109 wierszy, 16 kolumn
Zbiór treningowy: (4087, 15), Zbiór testowy: (1022, 15)
Liczba udarów w zbiorze testowym: 50 z 1022 pacjentów


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.utils.class_weight import compute_class_weight

# Ustawienie ziarna losowości dla powtarzalności wyników
tf.random.set_seed(42)

# 1. Obliczenie wag klas (rekompensata za silne niezbalansowanie 95% vs 5%)
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Wagi klas przekazane do sieci:", class_weights)

# 2. Budowa płytkiej sieci neuronowej (MLP)
model_mlp = Sequential([
    Input(shape=(X_train.shape[1],)),       # Warstwa wejściowa (15 cech)
    Dense(32, activation="relu"),           # 1. warstwa ukryta
    Dropout(0.2),                           # Regularyzacja przeciw przeuczeniu
    Dense(16, activation="relu"),           # 2. warstwa ukryta
    Dense(1, activation="sigmoid")          # Wyjście binarne (prawdopodobieństwo udaru)
])

model_mlp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Recall(name="recall")]
)

model_mlp.summary()

# 3. Trening sieci
history = model_mlp.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

Wagi klas przekazane do sieci: {np.int64(0): np.float64(0.5255915637860082), np.int64(1): np.float64(10.268844221105528)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,057 (4.13 KB)

 Trainable params: 1,057 (4.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5616 - loss: 0.6208 - recall: 0.7697 - val_accuracy: 0.7262 - val_loss: 0.5402 - val_recall: 0.7447
Epoch 2/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6840 - loss: 0.5297 - recall: 0.7961 - val_accuracy: 0.7372 - val_loss: 0.4688 - val_recall: 0.7234
Epoch 3/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6916 - loss: 0.4938 - recall: 0.8026 - val_accuracy: 0.7482 - val_loss: 0.4426 - val_recall: 0.7234
Epoch 4/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7146 - loss: 0.4739 - recall: 0.8421 - val_accuracy: 0.7482 - val_loss: 0.4446 - val_recall: 0.7234
Epoch 5/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7066 - loss: 0.4792 - recall: 0.8421 - val_accuracy: 0.7482 - val_loss: 0.4337 - val_recall: 0.7234
Epoch 6/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7222 - loss: 0.4619 - recall: 0.8882 - val_accuracy: 0.7579 - val_loss: 0.4217 - val_recall: 0.7234
Epoc